In [3]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import pandas as pd
import numpy as np
import transport_analytics as ta

# Apply a professional style
sns.set_theme(style="whitegrid", context="paper")

def visualize_timeline(hub_name, trains, bikes, config, zoom_start, zoom_end, event_type='arrival'):
    """
    PLOTS THE HEARTBEAT (Timeline View)
    Shows specific trains and the resulting bike rental pressure.
    """
    print(f"Generating Heartbeat Plot for {hub_name}...")

    # 1. SETUP
    start = pd.to_datetime(zoom_start)
    end = pd.to_datetime(zoom_end)
    timeline = pd.date_range(start, end, freq='1min')

    # 2. FILTER DATA
    # Filter Trains
    time_col = 'actual_arrival' if event_type == 'arrival' else 'actual_departure'
    t_mask = (trains[time_col] >= start) & (trains[time_col] <= end)
    zoom_trains = trains[t_mask].copy()

    # Filter Bikes
    bike_col = 'start_time' if event_type == 'arrival' else 'end_time'
    b_mask = (bikes[bike_col] >= start) & (bikes[bike_col] <= end)
    zoom_bikes = bikes[b_mask].copy()

    # 3. GENERATE THEORETICAL PRESSURE
    # We call the engine from transport_analytics
    sig_arr, sig_dep = ta.generate_signals(zoom_trains, timeline, config['params'])
    pressure_signal = sig_arr if event_type == 'arrival' else sig_dep

    # 4. GET ACTUAL BIKE COUNTS
    bike_counts = zoom_bikes.set_index(bike_col).resample('1min').size().reindex(timeline, fill_value=0)

    # --- PLOTTING ---
    fig, ax1 = plt.subplots(figsize=(14, 6), dpi=120)

    # A. Actual Rentals (Blue Bars)
    # Using a slightly transparent blue for readability
    ax1.bar(timeline, bike_counts, width=0.0007, color='#004c6d', alpha=0.5, label='Actual Rentals', zorder=2)
    ax1.set_ylabel('Bikes Rented per Minute', color='#004c6d', fontweight='bold')
    ax1.tick_params(axis='y', labelcolor='#004c6d')
    ax1.set_ylim(0, max(bike_counts.max() * 1.3, 1)) # Scale y-axis dynamically

    # B. Theoretical Pressure (Orange Line)
    ax2 = ax1.twinx()
    color_sig = '#d55e00' # Orange for Arrival
    if event_type == 'departure': color_sig = '#009e73' # Green for Departure

    ax2.plot(timeline, pressure_signal, color=color_sig, linewidth=2.5, label='Passenger Pressure', zorder=3)
    ax2.fill_between(timeline, pressure_signal, color=color_sig, alpha=0.15, zorder=1)
    ax2.set_ylabel('Model Pressure (Probability)', color=color_sig, fontweight='bold')
    ax2.tick_params(axis='y', labelcolor=color_sig, labelsize=9)
    ax2.grid(False) # Turn off grid for secondary axis to avoid clutter

    # C. Train Markers (Bottom Axis)
    # We differentiate Long Distance vs Regional
    y_min, y_max = ax2.get_ylim()

    for _, t in zoom_trains.iterrows():
        t_time = t[time_col]
        t_cat = t.get('train_type', 'Other')

        # Style based on train type
        is_long = t_cat in ['IC', 'EC', 'Ex', 'R', 'rj', 'SC']
        marker_color = '#333333' if is_long else '#888888'
        line_style = '-' if is_long else ':'
        alpha = 0.8 if is_long else 0.5
        height = 0.15 if is_long else 0.10

        # Plot the vertical line marker
        ax2.vlines(x=t_time, ymin=0, ymax=y_max*height, color=marker_color,
                   linestyle=line_style, alpha=alpha, linewidth=1.5)

        # Add Label (Train Type)
        if is_long: # Only label big trains to avoid clutter
            ax2.text(t_time, y_max*(height+0.02), t_cat, ha='center', fontsize=7,
                     color=marker_color, rotation=90, fontweight='bold')

    # Formatting
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    ax1.xaxis.set_major_locator(mdates.HourLocator(interval=1))
    plt.title(f"{hub_name.replace('_', ' ')}: {event_type.title()} Analysis\n{start.strftime('%d %b %Y')}", fontsize=14, pad=15)

    # Unified Legend
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', frameon=True, framealpha=0.9)

    plt.tight_layout()
    plt.savefig(f"Viz_Timeline_{hub_name}_{event_type}.png")
    plt.show()

def visualize_average_pulse(hub_name, trains, bikes, config, window_min=-15, window_max=60):
    """
    PLOTS THE AVERAGE PULSE (Scientific View)
    Stacks all train arrivals on top of each other (Time=0) to see the average crowd behavior.
    """
    print(f"Generating Average Pulse for {hub_name}...")

    deltas = []

    # 1. Calculate Deltas
    # For every train, find bike rentals that happened nearby
    for t_arr in trains['actual_arrival']:
        start_w = t_arr + pd.Timedelta(minutes=window_min)
        end_w = t_arr + pd.Timedelta(minutes=window_max)

        # Find rentals in this window
        nearby = bikes[(bikes['start_time'] >= start_w) & (bikes['start_time'] <= end_w)]

        # Calculate diff
        for r_time in nearby['start_time']:
            diff = (r_time - t_arr).total_seconds() / 60
            deltas.append(diff)

    if not deltas:
        print("  Warning: No overlapping data found for pulse.")
        return

    # 2. Plotting
    df_pulse = pd.DataFrame(deltas, columns=['minutes'])

    plt.figure(figsize=(10, 6), dpi=120)

    # Kernel Density Estimate (The Smooth Curve)
    sns.kdeplot(data=df_pulse, x='minutes', bw_adjust=0.5, fill=True,
                color='#004c6d', alpha=0.3, linewidth=2, label='Avg Rental Density')

    # Add the "Expected Peak" from your Config
    expected_peak = config['params']['peak_arr']
    plt.axvline(expected_peak, color='#d55e00', linestyle='--', linewidth=2, label=f'Model Peak ({expected_peak}m)')

    # Reference Line (Train Arrival)
    plt.axvline(0, color='black', linewidth=1.5, label='Train Arrival (t=0)')

    # Styling
    plt.title(f"Average Passenger Wave: {hub_name}", fontsize=14, fontweight='bold')
    plt.xlabel("Minutes relative to Train Arrival", fontsize=11)
    plt.ylabel("Rental Probability Density", fontsize=11)
    plt.xlim(window_min, window_max)
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f"Viz_Pulse_{hub_name}.png")
    plt.show()

In [ ]:
def visualize_results(results_df):
    """
    Takes the results DataFrame from run_master_analysis() and plots the comparison.
    """
    # Filter for just one metric (e.g., Arrival Correlation) to keep it clean
    df_plot = results_df[results_df['Category'].isin(['Regional', 'LongDist'])].copy()

    plt.figure(figsize=(12, 6))

    # Grouped Bar Chart
    sns.barplot(data=df_plot, x='Hub', y='Arr_Corr', hue='Category', palette='viridis')

    plt.axhline(0, color='black', linewidth=1)
    plt.title("Hypothesis Check: Do Regional Trains Drive More Demand?", fontsize=14)
    plt.ylabel("Correlation Strength (Pearson r)", fontsize=12)
    plt.xlabel("City / Hub", fontsize=12)
    plt.grid(axis='y', alpha=0.3)
    plt.legend(title='Train Type')

    plt.show()

In [2]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

def plot_average_pulse(trains, bikes, city_name, window_min=-20, window_max=60):
    """
    Stacks all train arrivals to find the 'Average Passenger Wave'.
    """
    # 1. Setup relative timeline
    deltas = []

    # For every train, find bike rentals that happened nearby
    for t_arr in trains['actual_arrival']:
        # Define window around this specific train
        start_w = t_arr + pd.Timedelta(minutes=window_min)
        end_w = t_arr + pd.Timedelta(minutes=window_max)

        # Find rentals in this window
        nearby_rentals = bikes[(bikes['start_time'] >= start_w) & (bikes['start_time'] <= end_w)]

        # Calculate "Minutes After Train" for each rental
        for r_time in nearby_rentals['start_time']:
            diff_mins = (r_time - t_arr).total_seconds() / 60
            deltas.append(diff_mins)

    # 2. Create the Plot Data
    df_pulse = pd.DataFrame(deltas, columns=['minutes_relative'])

    # 3. Plot Density (The Wave)
    plt.figure(figsize=(10, 6))
    sns.histplot(data=df_pulse, x='minutes_relative', binwidth=2, kde=True,
                 color='#004c6d', alpha=0.6, element="step")

    # Styling
    plt.axvline(0, color='#d55e00', linestyle='--', linewidth=2, label='Train Arrival')
    plt.title(f"Average Passenger Response: {city_name}", fontsize=14, fontweight='bold')
    plt.xlabel("Minutes relative to arrival (0 = Train Arrived)")
    plt.ylabel("Rental Frequency")
    plt.xlim(window_min, window_max)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"Average_Pulse_{city_name}.png", dpi=300) # High Res for thesis
    plt.show()